In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset"

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        path = os.path.join(root, file)
        print(f"{file} | {os.path.getsize(path) / (1024**2):.2f} MB")

print("\n" + "=" * 50)


X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"), mmap_mode="r")
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"), mmap_mode="r")
X_val = np.load(os.path.join(DATA_DIR, "X_val.npy"), mmap_mode="r")
y_val = np.load(os.path.join(DATA_DIR, "y_val.npy"), mmap_mode="r")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("\nData type:", X_train.dtype)
print("Pixel value range:", X_train.min(), "to", X_train.max())
print("Training samples:", len(y_train))
print("Validation samples:", len(y_val))

class_names = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}

distribution = pd.DataFrame({
    "Class": list(class_names.values()),
    "Train Count": [
        np.sum(y_train == i) for i in range(5)
    ],
    "Validation Count": [
        np.sum(y_val == i) for i in range(5)
    ]
})

display(distribution)

y_train.npy | 0.03 MB
y_val.npy | 0.01 MB
X_val.npy | 420.90 MB
X_train.npy | 2577.67 MB

X_train shape: (4489, 224, 224, 3)
y_train shape: (4489,)
X_val shape: (733, 224, 224, 3)
y_val shape: (733,)

Data type: float32
Pixel value range: 0.0 to 1.0
Training samples: 4489
Validation samples: 733


,Class,Train Count,Validation Count
0,No DR,1444,361
1,Mild,296,74
2,Moderate,799,200
3,Severe,770,39
4,Proliferative DR,1180,59


In [2]:

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

DATA_DIR = "/kaggle/input/datasets/omarashour206/aptos2019-preprocessed-dataset"
MODEL_PATH = "/kaggle/working/best_binary_dr_model.keras"
BATCH_SIZE = 32
EPOCHS = 10

print("GPU:", tf.config.list_physical_devices("GPU"))

X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"), mmap_mode="r")
y_train = np.load(os.path.join(DATA_DIR, "y_train.npy")).astype(np.int32)
X_val = np.load(os.path.join(DATA_DIR, "X_val.npy"), mmap_mode="r")
y_val = np.load(os.path.join(DATA_DIR, "y_val.npy")).astype(np.int32)

y_train = (y_train > 0).astype(np.float32)
y_val = (y_val > 0).astype(np.float32)

print("Train:", X_train.shape, np.bincount(y_train.astype(int)))
print("Validation:", X_val.shape, np.bincount(y_val.astype(int)))

def make_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (np.arange(len(y)), y)
    )

    if training:
        ds = ds.shuffle(len(y))

    def load_batch(indices, labels):
        images = tf.numpy_function(
            lambda idx: np.asarray(X[idx], dtype=np.float32) * 255.0,
            [indices],
            tf.float32
        )
        images.set_shape([None, 224, 224, 3])
        labels = tf.cast(labels, tf.float32)
        return images, labels

    ds = ds.batch(BATCH_SIZE).map(
        load_batch, num_parallel_calls=1
    )
    return ds.prefetch(1)

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train.astype(int)
)
class_weights = {0: float(weights[0]), 1: float(weights[1])}
print("Class weights:", class_weights)

base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)
model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

callbacks = [
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    )
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

model = tf.keras.models.load_model(MODEL_PATH)
probabilities = model.predict(val_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(
    y_val.astype(int),
    predictions,
    labels=[0, 1],
    target_names=["No DR", "DR"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_val.astype(int), predictions))
print("\nSaved model:", MODEL_PATH)


GPU: []
Train: (4489, 224, 224, 3) [1444 3045]
Validation: (733, 224, 224, 3) [361 372]
Class weights: {0: 1.5543628808864265, 1: 0.7371100164203612}


2026-10-19 12:47:03.075014: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7337 - auc: 0.8056 - loss: 0.5057
Epoch 1: val_auc improved from None to 0.97029, saving model to /kaggle/working/best_binary_dr_model.keras

Epoch 1: finished saving model to /kaggle/working/best_binary_dr_model.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 212s 1s/step - accuracy: 0.8614 - auc: 0.9448 - loss: 0.3503 - val_accuracy: 0.9195 - val_auc: 0.9703 - val_loss: 0.2402
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9467 - auc: 0.9855 - loss: 0.1812
Epoch 2: val_auc improved from 0.97029 to 0.97701, saving model to /kaggle/working/best_binary_dr_model.keras

Epoch 2: finished saving model to /kaggle/working/best_binary_dr_model.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 198s 1s/step - accuracy: 0.9528 - auc: 0.9878 - loss: 0.1644 - val_accuracy: 0.9332 - val_auc: 0.9770 - val_loss: 0.2034
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9577 - auc: 0